# Part 1: Build knowledge bases with agentic retrieval

You've just received your Zava access badge and laptop. Your first task: build the AI-powered knowledge system that the rest of the company will use to answer HR and benefits questions.

**🎯 Mission**
- Upload a cloud architecture PDF directly to a **File Knowledge Source** and query it
- Connect two pre-built search indexes (**HR docs** and **health benefits docs**) as **Indexed Knowledge Sources**
- Build a multi-source **Knowledge Base** that routes each sub-question to the right source automatically


## The building blocks

In Foundry IQ (Azure AI Search), a **knowledge base** is a top-level resource that connects a chat completion model to one or more knowledge sources. It defines which data sources to query, which model to use for reasoning, and how query execution should be optimized.

In this notebook, you'll explore two types of knowledge sources:

1. **File Knowledge Source**: Upload raw files directly. The service handles chunking, embedding, and indexing automatically.
2. **Indexed Knowledge Sources**: Connect to pre-built search indexes for production workloads with full control over schema and processing.

## Pre-configured environment

This lab uses the following Azure resources, already set up for you:

* Azure AI Search
  * `hrdocs` index: HR policies, handbook, company info
  * `healthdocs` index: health benefits, insurance, coverage
  * Semantic ranking enabled, pre-computed embeddings
* Azure OpenAI from Microsoft Foundry Models
  * `gpt-5.4`: Chat completion for reasoning and synthesis
  * `text-embedding-3-large`: Embedding model for vector search
* Microsoft Fabric

## Step 1: Set up environment variables

Load the configuration for your Azure resources. The `.env` file in the project folder has everything you need: Azure AI Search endpoints, Azure OpenAI credentials, and model configuration.

**ℹ️ Note**
- The first time you run the cell below, you'll be prompted to select a kernel. Select **Python Environments** and choose the **.venv** environment.
- You may also be prompted with "Do you want to allow public and private networks to access this app?" Select **Allow**.

**⚠️ Troubleshooting:**
If code cells get stuck and keep spinning, try these remediation steps:
1. Select **Restart** from the notebook toolbar at the top. If that doesn't work:
2. Select **Reload window** from the VS Code command palette. If that doesn't work:
3. Close VS Code completely and restart it.

In [ ]:
import json
import os

from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv

load_dotenv(override=True)

AZURE_SEARCH_SERVICE_ENDPOINT = os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"]
AZURE_SEARCH_ADMIN_KEY = os.environ["AZURE_SEARCH_ADMIN_KEY"]
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_SEARCH_API_VERSION = "2026-05-01-preview"
AZURE_OPENAI_KEY = os.environ["AZURE_OPENAI_KEY"]
AZURE_OPENAI_CHATGPT_DEPLOYMENT = os.environ["AZURE_OPENAI_CHATGPT_DEPLOYMENT"]
AZURE_OPENAI_CHATGPT_MODEL_NAME = os.environ["AZURE_OPENAI_CHATGPT_MODEL_NAME"]
AZURE_OPENAI_EMBEDDING_DEPLOYMENT = os.environ.get("AZURE_OPENAI_EMBEDDING_DEPLOYMENT", "text-embedding-3-large")
AZURE_OPENAI_EMBEDDING_MODEL = "text-embedding-3-large"

HRDOCS_INDEX = "hrdocs"
HEALTHDOCS_INDEX = "healthdocs"

search_credential = AzureKeyCredential(AZURE_SEARCH_ADMIN_KEY)
print("Environment variables loaded")


---

## A: File Knowledge Source

A **File Knowledge Source** lets you upload raw files directly. The service handles chunking, embedding, and indexing automatically. This is the fastest way to get documents into a knowledge base.

## Step 2: Create a File Knowledge Source

Configure a file knowledge source with an embedding model for automatic ingestion. The `ingestion_parameters` specify:
- **Embedding model**: Generates vector embeddings for the uploaded content
- **Content extraction mode**: How to extract text from documents

In [ ]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    AzureOpenAIVectorizerParameters,
    FileKnowledgeSource,
    FileKnowledgeSourceParameters,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeSourceAzureOpenAIVectorizer,
    KnowledgeSourceIngestionParameters,
)

FILE_KNOWLEDGE_SOURCE_NAME = "file-docs-knowledge-source"

index_client = SearchIndexClient(endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, credential=search_credential)

file_ks = FileKnowledgeSource(
    name=FILE_KNOWLEDGE_SOURCE_NAME,
    description="Uploaded HR and company documents for Zava",
    file_parameters=FileKnowledgeSourceParameters(
        ingestion_parameters=KnowledgeSourceIngestionParameters(
            embedding_model=KnowledgeSourceAzureOpenAIVectorizer(
                azure_open_ai_parameters=AzureOpenAIVectorizerParameters(
                    resource_url=AZURE_OPENAI_ENDPOINT,
                    deployment_name=AZURE_OPENAI_EMBEDDING_DEPLOYMENT,
                    model_name=AZURE_OPENAI_EMBEDDING_MODEL,
                    api_key=AZURE_OPENAI_KEY,
                )
            ),
            content_extraction_mode="minimal",
            )
    ),
)

index_client.create_or_update_knowledge_source(file_ks)
print(f"File knowledge source '{FILE_KNOWLEDGE_SOURCE_NAME}' created or updated successfully.")

## Step 3: Upload files to the knowledge source

Upload a document directly to the file knowledge source. The service will automatically extract text, chunk it into searchable segments, and generate vector embeddings.

⏱️ The sample file takes around 40 seconds to ingest.

In [ ]:
from pathlib import Path

data_dir = Path("..") / "data" / "ai-search-data"
files_to_upload = [
    data_dir / "filesource" / "MSFT_cloud_architecture_zava.pdf"
]

uploaded_files = []
for file_path in files_to_upload:
    with open(file_path, "rb") as f:
        file_content = f.read()
    uploaded = index_client.upload_knowledge_source_file(
        FILE_KNOWLEDGE_SOURCE_NAME, file_content, filename=file_path.name
    )
    uploaded_files.append(uploaded)
    print(f"Uploaded: {file_path.name} (file_id: {uploaded.file_id}, {uploaded.file_size_bytes} bytes)")

print(f"\nTotal files uploaded: {len(uploaded_files)}")

## Step 4: Create a knowledge base for the file source

A knowledge base is the orchestration layer that combines:

1. Your data sources (the file knowledge source)
2. An LLM (Azure OpenAI) for understanding queries and generating answers
3. Configuration for how to process queries and format responses

This knowledge base uses `ANSWER_SYNTHESIS` output mode, so the attached LLM will generate a complete natural language answer from the retrieved chunks.

In [ ]:
from azure.search.documents.indexes.models import (
    KnowledgeBase,
    KnowledgeBaseAzureOpenAIModel,
    KnowledgeSourceReference,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeRetrievalOutputMode,
)

FILE_KNOWLEDGE_BASE_NAME = "file-docs-knowledge-base"

aoai_params = AzureOpenAIVectorizerParameters(
    resource_url=AZURE_OPENAI_ENDPOINT,
    deployment_name=AZURE_OPENAI_CHATGPT_DEPLOYMENT,
    model_name=AZURE_OPENAI_CHATGPT_MODEL_NAME,
    api_key=AZURE_OPENAI_KEY,
)

knowledge_base = KnowledgeBase(
    name=FILE_KNOWLEDGE_BASE_NAME,
    models=[KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=aoai_params)],
    knowledge_sources=[
        KnowledgeSourceReference(name=FILE_KNOWLEDGE_SOURCE_NAME),
    ],
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
)

index_client.create_or_update_knowledge_base(knowledge_base)
print(f"Knowledge base '{FILE_KNOWLEDGE_BASE_NAME}' created or updated successfully.")

## Step 5: Query the file knowledge base

Ask a question about the uploaded document. The knowledge base will search across the file content and synthesize a natural language answer with citations.

In [ ]:
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    FileKnowledgeSourceParams,
    KnowledgeBaseMessage,
    KnowledgeBaseMessageTextContent,
    KnowledgeBaseRetrievalRequest,
)
from IPython.display import Markdown, display

file_kb_client = KnowledgeBaseRetrievalClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, knowledge_base_name=FILE_KNOWLEDGE_BASE_NAME, credential=search_credential
)

file_ks_params = FileKnowledgeSourceParams(
    knowledge_source_name=FILE_KNOWLEDGE_SOURCE_NAME,
    include_references=True,
    include_reference_source_data=True,
)

question = "As a new Zava engineer, what cloud architecture principles should I know about? What data sensitivity levels does Zava use?"

req = KnowledgeBaseRetrievalRequest(
    messages=[
        KnowledgeBaseMessage(
            role="user",
            content=[
                KnowledgeBaseMessageTextContent(text=question)
            ],
        )
    ],
    knowledge_source_params=[file_ks_params],
    include_activity=True,
)

result = file_kb_client.retrieve(retrieval_request=req)
display(Markdown(result.response[0].content[0].text))

### Review the activity log

The activity log shows the sequence of steps taken by the knowledge base. Each activity contains:

* `type`: The type of activity. For this log, you will see the first step of "modelQueryPlanning", multiple "file" steps (one for each query sent to the file knowledge source), "modelAnswerSynthesis" for the answer synthesis step, and "agenticReasoning" for the semantic re-ranking step.
* `elapsedMS`: The duration of the activity.
* `id`: The activity ID, which is helpful to connect each reference to the activity that produced it. You'll review references next.

The activity entries also contain other metadata specific to their type, so that you can see how many tokens were used by any model-powered steps and see the queries sent to sources for retrieval steps.

In [ ]:
activities_json = [activity.as_dict() for activity in result.activity or []]
print(json.dumps(activities_json, indent=2))

### Review the references

The next cell prints the raw `references` array returned by the knowledge base.

The references show which sources were retrieved, and the source metadata for each reference. Every result contains:

* `type`: Always "file", since this knowledge base only contains a file source.
* `id`: A numeric identifier for the reference. This is what appears in the answer as citation markers like `[ref:1]`.
* `activitySource`: The id of the retrieval activity in the activity log that produced this reference. You can use this to trace which search step returned this document.
* `sourceData`: Fields from the retrieved chunk in the auto-created search index - always `uid`, `snippet`, and `metadata_storage_path` for file source indexes.
* `rerankerScore`: A score between 0-4 from the re-ranking model, where 4 is the best.
* `docName`: The auto-generated filename for the uploaded file.

In [ ]:
references_json = [ref.as_dict() for ref in result.references or []]
print(json.dumps(references_json, indent=2))

---

## B: Indexed Knowledge Sources

**Indexed Knowledge Sources** connect to pre-built search indexes, giving you full control over schema, chunking, and embedding. This is ideal for production workloads with large datasets.

For this section, we already pre-populated two search indexes with ingested document chunks. Typically, you would set up an indexer pipeline to ingest documents into the search index from a source like Azure Blob Storage.

## Step 6: Verify search indexes

Run the cell below to confirm each index has the expected number of document chunks:

* `hrdocs`: 50 document chunks about HR policies
* `healthdocs`: 334 document chunks about health benefits and insurance

In [ ]:
from azure.search.documents import SearchClient

for index_name in [HRDOCS_INDEX, HEALTHDOCS_INDEX]:
    client = SearchClient(AZURE_SEARCH_SERVICE_ENDPOINT, index_name, search_credential)
    count = client.get_document_count()
    print(f"{index_name}: {count} documents")


## Step 7: Create two indexed knowledge sources

You'll create two knowledge sources pointing to each index.

* `healthdocs-knowledge-source`: Points to the `healthdocs` index
* `hrdocs-knowledge-source`: Points to the `hrdocs` index

Both sources set the `snippet` field as the searchable field, and include `blob_path` as a source data field, so that the path can be used in citation links.

In [ ]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndexFieldReference,
    SearchIndexKnowledgeSource,
    SearchIndexKnowledgeSourceParameters,
)

HR_KNOWLEDGE_SOURCE_NAME = "hrdocs-knowledge-source"
HEALTH_KNOWLEDGE_SOURCE_NAME = "healthdocs-knowledge-source"
KNOWLEDGE_SOURCE_NAMES = [HR_KNOWLEDGE_SOURCE_NAME, HEALTH_KNOWLEDGE_SOURCE_NAME]

index_client = SearchIndexClient(endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, credential=search_credential)

for knowledge_source_name, index_name, description in [
    (HR_KNOWLEDGE_SOURCE_NAME, HRDOCS_INDEX, "Zava HR documents"),
    (HEALTH_KNOWLEDGE_SOURCE_NAME, HEALTHDOCS_INDEX, "Zava health benefits documents"),
]:
    knowledge_source = SearchIndexKnowledgeSource(
        name=knowledge_source_name,
        description=description,
        search_index_parameters=SearchIndexKnowledgeSourceParameters(
            search_index_name=index_name,
            source_data_fields=[
                SearchIndexFieldReference(name="blob_path"),
                SearchIndexFieldReference(name="snippet"),
            ],
            search_fields=[SearchIndexFieldReference(name="snippet")],
            semantic_configuration_name="semantic-configuration",
        ),
    )
    index_client.create_or_update_knowledge_source(knowledge_source=knowledge_source)
    print(f"Knowledge source '{knowledge_source.name}' created or updated successfully.")


## Step 8: Create the multi-source knowledge base

Create a knowledge base that contains both search index knowledge sources. Like the previous knowledge base, this one uses `output_mode=ANSWER_SYNTHESIS`, so it will use the attached LLM to answer the original query with grounded citations.

In [ ]:
from azure.search.documents.indexes.models import (
    AzureOpenAIVectorizerParameters,
    KnowledgeBase,
    KnowledgeBaseAzureOpenAIModel,
    KnowledgeSourceReference,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeRetrievalLowReasoningEffort,
    KnowledgeRetrievalOutputMode,
)

KNOWLEDGE_BASE_NAME = "multisource-search-knowledge-base"

aoai_params = AzureOpenAIVectorizerParameters(
    resource_url=AZURE_OPENAI_ENDPOINT,
    deployment_name=AZURE_OPENAI_CHATGPT_DEPLOYMENT,
    model_name=AZURE_OPENAI_CHATGPT_MODEL_NAME,
    api_key=AZURE_OPENAI_KEY,
)

knowledge_base = KnowledgeBase(
    name=KNOWLEDGE_BASE_NAME,
    description="Multi-source knowledge base over HR and health document indexes",
    models=[KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=aoai_params)],
    knowledge_sources=[KnowledgeSourceReference(name=name) for name in KNOWLEDGE_SOURCE_NAMES],
    retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
)

index_client.create_or_update_knowledge_base(knowledge_base)
print(f"Knowledge base '{KNOWLEDGE_BASE_NAME}' created or updated successfully.")

## Step 9: Query the multi-source knowledge base

Let's ask a two-part onboarding question that spans both knowledge sources:

* _"How many vacation weeks does a senior Zava employee get?"_ → HR-related, should come from `hrdocs`
* _"Which health plan gives the best coverage for mental health services?"_ → Benefits-related, should come from `healthdocs`

The knowledge base uses agentic retrieval to:

1. Analyze the query and recognize two distinct topics
2. Decompose it into focused sub-queries
3. Select the relevant knowledge source for each sub-query
4. Run searches concurrently across both sources
5. Merge and return the results

In [ ]:
from azure.search.documents.knowledgebases.models import (
    SearchIndexKnowledgeSourceParams,
)

knowledge_base_client = KnowledgeBaseRetrievalClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, knowledge_base_name=KNOWLEDGE_BASE_NAME, credential=search_credential
)

hrdocs_knowledge_source_params = SearchIndexKnowledgeSourceParams(
    knowledge_source_name=HR_KNOWLEDGE_SOURCE_NAME,
    include_references=True,
    include_reference_source_data=True,
)
healthdocs_knowledge_source_params = SearchIndexKnowledgeSourceParams(
    knowledge_source_name=HEALTH_KNOWLEDGE_SOURCE_NAME,
    include_references=True,
    include_reference_source_data=True,
)

question = (
    "I just joined Zava as a senior engineer. "
    "According to company policy, how many vacation weeks does a senior employee get? "
    "And among the available health plans, which gives the best coverage for mental health services?"
)

retrieval_request = KnowledgeBaseRetrievalRequest(
    messages=[
        KnowledgeBaseMessage(
            role="user",
            content=[KnowledgeBaseMessageTextContent(text=question)],
        )
    ],
    knowledge_source_params=[hrdocs_knowledge_source_params, healthdocs_knowledge_source_params],
    include_activity=True,
)

result = knowledge_base_client.retrieve(retrieval_request=retrieval_request)
display(Markdown(result.response[0].content[0].text))

### Review the activity log

For this activity log, you will see multiple "searchIndex" steps, one for each query sent to each search index.

In [ ]:
activities_json = [activity.as_dict() for activity in result.activity or []]
print(json.dumps(activities_json, indent=2))

### Review the references

This time, the references have a type of "searchIndex" and the "sourceData" fields are slightly different, since these search indexes have a different field configuration than the auto-created file source index.

In [ ]:
references_json = [ref.as_dict() for ref in result.references or []]
print(json.dumps(references_json, indent=2))

#### 🔍 Source Hunt

Look through the reference and activity log above. Can you find:

1. Which knowledge source answered the **vacation** question?
2. Which knowledge source answered the **mental health plan** question?
3. Did the agent query **both** sources, or only the most relevant one for each sub-query?

## Bonus: Query from the Copilot CLI

Every knowledge base exposes an MCP server endpoint, which can be added to any MCP-compatible client like VS Code or GitHub Copilot CLI.
If you want to try out querying the knowledge base from the Copilot CLI, go through the set up steps in [Copilot CLI sidequest](copilot-cli-sidequest.md) and run the commands below to use the Zava knowledge base as an MCP server.

In [ ]:
mcp_url = f"{AZURE_SEARCH_SERVICE_ENDPOINT}/knowledgebases/{KNOWLEDGE_BASE_NAME}/mcp?api-version={AZURE_SEARCH_API_VERSION}"
print(f"copilot mcp add zava-kb \"{mcp_url}\" --header \"api-key={AZURE_SEARCH_ADMIN_KEY}\"")
print("copilot -i \"I just joined Zava as a senior engineer. According to company policy, how many vacation weeks does a senior employee get?\"")

## ✅ Mission complete

**What you built:**

* ✓ **File Knowledge Source**: A source that supports direct PDF uploading and handles all the chunking and embedding automatically.
* ✓ **Indexed Knowledge Sources**: A source for pre-built HR and health search indexes, so that you can have full control over the index schema and processing.
* ✓ **Multi-source Knowledge Base**: A single KB that routes each sub-question to the right source automatically, with citation-backed answers.

➡️ Continue to [Part 2: Add grounding from web results](part2-search-mcp-kb.ipynb).